# Session 6 — TPU: JAX/Flax Throughput Benchmark

## Background

Sessions 1–5 reached the TPU via PyTorch/XLA — a minimal-change path that preserves existing PyTorch code but uses XLA tracing under the hood. JAX takes a different approach: functional programming model, explicit parameter trees, and `jax.jit` compiling the full train step into a single XLA program. In principle, both paths compile to the same XLA HLO representation on the TPU. Session 6 tests whether that principle holds in practice on the v5litepod-1.

Session 1 measured PyTorch/XLA TPU throughput as the baseline. This notebook measures JAX/Flax throughput on the same hardware using the equivalent architecture (`FlaxTransformerBlock` in `flax_model.py`). Session 6's analysis will overlay the two curves.

## Goal

Measure JAX/Flax training throughput on the TPU v5litepod-1 across the same batch sweep as Session 1. Record first-step compilation time per batch size. Produce `results/tpu_jax.json` for comparison in `03-analysis.ipynb`.

## Question

Does switching from PyTorch/XLA to JAX/Flax change TPU throughput, or do both frameworks converge to the same performance through XLA?

---

**Runtime:** TPU (v5litepod-1 or equivalent)

After running, results are saved to `results/tpu_jax.json` and loaded automatically by `03-analysis.ipynb`.

In [ ]:
# Install JAX with TPU support, Flax, and Optax.
import importlib, subprocess, sys

def _need(pkg):
    try:
        importlib.import_module(pkg)
        return False
    except ImportError:
        return True

if _need("jax"):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "jax[tpu]", "flax", "optax",
        "-f", "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
    ])
    print("Installed jax[tpu], flax, optax")
else:
    print("JAX already available — skipping install")

In [ ]:
import jax
import jax.numpy as jnp
import flax
import optax
import time

tpu_devices = jax.devices("tpu")
assert len(tpu_devices) > 0, "No TPU found. Ensure this notebook is running on a TPU runtime."

device = tpu_devices[0]
print(f"JAX version  : {jax.__version__}")
print(f"Flax version : {flax.__version__}")
print(f"Optax version: {optax.__version__}")
print(f"TPU device   : {device}")
print(f"All devices  : {jax.devices()}")

In [ ]:
import sys; sys.path.insert(0, ".")
from flax_model import FlaxTransformerBlock, D_MODEL, N_HEAD, DIM_FEEDFORWARD, SEQ_LEN

# ── Session 6 config ──────────────────────────────────────────────────────────
BATCH_SIZES = [4, 8, 16, 32, 64, 128, 256, 512, 1024]   # mirror Session 1
N_STEPS     = 50
WARMUP      = 5

model     = FlaxTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)
optimizer = optax.adam(1e-4)

# ── JIT-compiled train step ───────────────────────────────────────────────────
# On TPU, jax.jit compiles to XLA HLO the first time it is called for a given
# input shape. The compilation spike is the same phenomenon as PyTorch/XLA's
# first xm.mark_step() call. Session 6's analysis will compare the two.
@jax.jit
def train_step(params, opt_state, x, y):
    def loss_fn(p):
        pred = model.apply(p, x)
        return jnp.mean((pred - y) ** 2)
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, new_opt_state = optimizer.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    return new_params, new_opt_state, loss

print(f"Model        : FlaxTransformerBlock  d_model={D_MODEL}  nhead={N_HEAD}  ff={DIM_FEEDFORWARD}")
print(f"Seq len      : {SEQ_LEN}")
print(f"Batch sizes  : {BATCH_SIZES}")
print(f"Steps        : {N_STEPS} (+ {WARMUP} warmup)")
print("train_step defined (first call per batch size will include XLA compilation).")

In [ ]:
def benchmark(batch_size, n_steps, warmup):
    """Benchmark one batch size.  Returns (compile_time_s, throughput, latency_ms)."""
    init_key = jax.random.PRNGKey(42)
    dummy_x  = jnp.ones((batch_size, SEQ_LEN, D_MODEL))
    params    = model.init(init_key, dummy_x)
    opt_state = optimizer.init(params)

    rng = jax.random.PRNGKey(0)

    # ── Step 0: first call = XLA compilation + execution ─────────────────────
    rng, k1, k2 = jax.random.split(rng, 3)
    x = jax.random.normal(k1, (batch_size, SEQ_LEN, D_MODEL))
    y = jax.random.normal(k2, (batch_size, SEQ_LEN, D_MODEL))

    t0 = time.perf_counter()
    params, opt_state, loss = train_step(params, opt_state, x, y)
    jax.block_until_ready(loss)
    compile_time = time.perf_counter() - t0

    # ── Warmup: remaining warmup-1 steps ──────────────────────────────────────
    for _ in range(warmup - 1):
        rng, k1, k2 = jax.random.split(rng, 3)
        x = jax.random.normal(k1, (batch_size, SEQ_LEN, D_MODEL))
        y = jax.random.normal(k2, (batch_size, SEQ_LEN, D_MODEL))
        params, opt_state, loss = train_step(params, opt_state, x, y)
        jax.block_until_ready(loss)

    # ── Timed steps ───────────────────────────────────────────────────────────
    elapsed = 0.0
    for _ in range(n_steps):
        rng, k1, k2 = jax.random.split(rng, 3)
        x = jax.random.normal(k1, (batch_size, SEQ_LEN, D_MODEL))
        y = jax.random.normal(k2, (batch_size, SEQ_LEN, D_MODEL))
        t0 = time.perf_counter()
        params, opt_state, loss = train_step(params, opt_state, x, y)
        jax.block_until_ready(loss)
        elapsed += time.perf_counter() - t0

    throughput = (n_steps * batch_size) / elapsed
    latency_ms = (elapsed / n_steps) * 1000
    return round(compile_time, 3), round(throughput, 1), round(latency_ms, 2)

print("Benchmark function defined.")

In [ ]:
results       = {}
compile_times = {}

print(f"{'batch':<8}  {'compile':>10}  {'throughput':>15}  {'latency':>10}")
print(f"{'─'*8}  {'─'*10}  {'─'*15}  {'─'*10}")

for bs in BATCH_SIZES:
    try:
        ctime, tput, lat = benchmark(bs, N_STEPS, WARMUP)
        results[str(bs)]       = {"throughput": tput, "latency_ms": lat}
        compile_times[str(bs)] = ctime
        print(f"{bs:<8}  {ctime:>9.2f}s  {tput:>14,.0f}  {lat:>9.1f} ms")
    except Exception as e:
        results[str(bs)]       = None
        compile_times[str(bs)] = None
        print(f"{bs:<8}  {'ERROR':>10}  {str(e)[:50]}")

print("\n── Compilation times (first call per shape) ─────────")
for bs, ct in compile_times.items():
    if ct is not None:
        print(f"  batch={bs:<6}: {ct:.2f} s")

In [ ]:
import json, os
from datetime import datetime, timezone

os.makedirs("results", exist_ok=True)
payload = {
    "hardware":        "TPU",
    "framework":       "JAX/Flax",
    "device_name":     str(jax.devices("tpu")[0]),
    "jax_version":     jax.__version__,
    "flax_version":    flax.__version__,
    "session":         "6",
    "batch_sizes":     BATCH_SIZES,
    "seq_len":         SEQ_LEN,
    "timestamp":       datetime.now(timezone.utc).isoformat(),
    "compile_times_s": compile_times,
    "results":         results,
}
path = "results/tpu_jax.json"
with open(path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved → {path}")